# Phase 7 — Scenario Engine Validation Notebook

This notebook validates the **production** `backend/simulation/scenarios.py`
module: the frozen simulation.md §3 schema, the clamp-within-1.25x/reject
bounds rule, the eight frozen §4 presets, the AI-candidate intake with
mapping notes, the FD-3 confirmation contract, and normalization to twin
fractions.

**No twin execution in this notebook** — `run_twin`/`simulate` are never
called. Phase 7 owns validation only; the twin owns simulation and
Phase 8 owns comparison. No network access, no API keys, no LLM calls:
AI-translated input is represented by hand-written candidate dicts.


In [ ]:
import sys
from pathlib import Path

# Path bootstrap so a fresh kernel can import the repository package.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "backend").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from backend.core.errors import AppError
from backend.simulation import scenarios as sc

print("scenario version:", sc.SCENARIO_VERSION)
print("presets:", [p["preset_id"] for p in sc.list_presets()])


## 1. A form scenario validates directly (FD-3: form runs without confirmation)

Positive FX here means company-currency *depreciation* (locked decision D):
import costs rise via κ_fx and foreign demand falls via ρ_rev·e_fx.

In [ ]:
form = sc.validate_params(
    {
        "name": "revenue dip + rate rise",
        "horizon_months": 12,
        "ramp_months": 0,
        "revenue_change_pct": -20.0,
        "interest_rate_change_pp": 2.0,
        "fx_change_pct": 10.0,
    },
    origin=sc.ScenarioOrigin.FORM,
)
print(form.confirmation_status.value, "| clamped:", form.clamped)
print(form.params.model_dump())
assert form.confirmation_status.value == "validated"


## 2. Bounds: one field in the clamp band, one beyond it

In [ ]:
clamped = sc.validate_params(
    {"name": "clamp demo", "revenue_change_pct": -60.0},
    origin=sc.ScenarioOrigin.FORM,
)
print("clamped:", clamped.clamped, "->", clamped.params.revenue_change_pct)
print(clamped.guardrail_events[0].model_dump())
assert clamped.params.revenue_change_pct == -50.0

try:
    sc.validate_params(
        {"name": "reject demo", "revenue_change_pct": -90.0},
        origin=sc.ScenarioOrigin.FORM,
    )
    raise AssertionError("should have rejected")
except AppError as exc:
    print("rejected with code:", exc.code)
    assert exc.code == "SCENARIO_OUT_OF_BOUNDS"


## 3. Presets: the frozen eight, with application defaults

In [ ]:
for entry in sc.list_presets():
    print(entry["preset_id"], "->", entry["params"])
combo = sc.get_preset("combined_stress")
print(combo.params.model_dump())
assert combo.params.name == "combined_stress"
assert combo.params.horizon_months == 12 and combo.params.ramp_months == 0


## 4. AI translation: candidate + mapping notes, awaiting confirmation

The translator is untrusted: its JSON is re-validated from scratch.
An explicit '200 basis points' quote normalizes to 2.0pp (locked
decision B) with the conversion recorded in `mapping_notes`.

In [ ]:
pp = sc.normalize_basis_points(200.0, note="rates +200bp -> +2pp")
candidate = sc.TranslationCandidate(
    candidate={
        "name": "nl scenario",
        "revenue_change_pct": -20.0,
        "interest_rate_change_pp": pp,
    },
    mapping_notes=[
        "revenue falls 20% -> revenue_change_pct -20",
        "rates +200bp -> interest_rate_change_pp 2.0",
    ],
    source_text="What if revenue falls 20% and rates rise 200bp?",
    model="notebook-double",
)
pending = sc.validate_candidate(candidate)
print(pending.confirmation_status.value, "|", pending.mapping_notes)
assert pending.confirmation_status.value == "awaiting_confirmation"


## 5. Ambiguity is never guessed (locked decision C)

'Rates rise 2%' could be +2pp or +2% relative — the rate field stays 0.0
and the scenario stays unconfirmed until a human resolves it.

In [ ]:
ambiguous = sc.validate_candidate(
    sc.TranslationCandidate(
        candidate={"name": "ambiguous rates"},
        mapping_notes=["AMBIGUOUS: rates rise 2% — pp or relative?"],
        source_text="rates rise 2%",
    )
)
print(ambiguous.confirmation_status.value, ambiguous.params.interest_rate_change_pp)
try:
    sc.to_overrides(ambiguous)
    raise AssertionError("should require confirmation")
except AppError as exc:
    print("conversion blocked with code:", exc.code)
    assert exc.code == "SCENARIO_PENDING_CONFIRM"


## 6. Confirm (re-validates) and normalize to twin fractions

In [ ]:
done = sc.confirm(pending)
print(done.confirmation_status.value, "| hash:", done.provenance.validated_params_hash)
overrides = sc.to_overrides(done)
print(overrides.model_dump())
assert overrides.revenue_change == -0.20
assert overrides.rate_change == 0.02
cancelled = sc.cancel(ambiguous)
print("cancelled:", cancelled.confirmation_status.value)


## 7. What this notebook does **not** claim

- No twin run was performed: deltas are *persistent monthly levels*
  (locked decision E), shown here only as normalized fractions.
- Presets are template defaults, not regulator-calibrated.
- `one_off_cost` has no upper bound by frozen design (locked decision K).
- Assumption-based intervention, not causal truth; not financial advice.